In [24]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"


# 1. Read data

In [25]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/iterative/"

In [26]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-A*02:01', 'HLA-A*03:01', 'HLA-A*01:01', 'HLA-B*07:02', 'HLA-B*27:05']

# 2. Train HMMS

In [27]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions


NUM_RUNS = 2
TARGET_LENGTH = 9
TARGET_LENGTHS= [9]

In [28]:
start_length = 1

## 2.1 Filter data

In [29]:
per_allele_per_kfold_per_length_binders_train_filtered = dict()
per_allele_per_kfold_per_length_binders_test_filtered = dict()

for allele_name in ALLELES:
    per_allele_per_kfold_per_length_binders_train_filtered[allele_name] = dict()
    per_allele_per_kfold_per_length_binders_test_filtered[allele_name] = dict()

    for split_num in per_allele_per_kfold_per_length_binders_train[allele_name].keys():
        per_allele_per_kfold_per_length_binders_train_filtered[allele_name][split_num] = dict()
        per_allele_per_kfold_per_length_binders_test_filtered[allele_name][split_num] = dict()

        for target_length in per_allele_per_kfold_per_length_binders_train[allele_name][split_num].keys():
            data_array = per_allele_per_kfold_per_length_binders_train[allele_name][split_num][target_length]
            per_allele_per_kfold_per_length_binders_train_filtered[allele_name][split_num][target_length] = np.array([s[:start_length] for s in data_array ])

            data_array = per_allele_per_kfold_per_length_binders_test[allele_name][split_num][target_length]
            per_allele_per_kfold_per_length_binders_test_filtered[allele_name][split_num][target_length] = np.array([s[:start_length] for s in data_array ])

## 2.2 Training process

In [30]:
target_length_for_training = start_length
total_length = 9
cutoff = 0.00005
train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][9])
## Stop threshold Hardcoded here
stop_threshold =  abs(cutoff * train_size * -22/ total_length  * target_length_for_training)
stop_threshold

2.9711000000000003

In [31]:
%%time
TARGET_LENGTHS = [9]
algorithm = 'viterbi'

per_name_models = dict()
per_allele_per_run_per_split_models=dict()
for allele_name in ALLELES:
    print(f"allele {allele_name}")
    per_allele_per_run_per_split_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        print(f"run {run_num}")

        print()


        per_kfold_per_length_data =  per_allele_per_kfold_per_length_binders_train_filtered[allele_name]

        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test_filtered[allele_name],
                                        verbose=True,
                                        multiple_check_input=False,
                                        algorithm=algorithm,
                                        name_prefix=f"{run_num}_run",
                                        num_states=4,
                                        n_jobs=10,
                                        stop_threshold=stop_threshold,
                                        min_iterations=5
                                        )
        per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)


allele HLA-A*02:01
run 0

split 0
train size 24309
Shuffling for fairness 
Simple training: Train Length 24309, init_batch_size None
[1] Improvement: 38983.803134316884	Time (s): 2.227
[2] Improvement: 0.0	Time (s): 2.077
[3] Improvement: 0.0	Time (s): 2.298
[4] Improvement: 0.0	Time (s): 2.296
[5] Improvement: 0.0	Time (s): 2.22
Total Training Improvement: 38983.803134316884
Total Training Time (s): 13.4276
run 1

split 0
train size 24309
Shuffling for fairness 
Simple training: Train Length 24309, init_batch_size None
[1] Improvement: 39562.60012165649	Time (s): 2.166
[2] Improvement: 0.0	Time (s): 2.193
[3] Improvement: 0.0	Time (s): 2.196
[4] Improvement: 0.0	Time (s): 2.173
[5] Improvement: 0.0	Time (s): 2.193
Total Training Improvement: 39562.60012165649
Total Training Time (s): 13.1575
allele HLA-A*03:01
run 0

split 0
train size 5132
Shuffling for fairness 
Simple training: Train Length 5132, init_batch_size None
[1] Improvement: 10083.575735181492	Time (s): 0.448
[2] Improveme

In [32]:
from hmm_logic_methods import add_more_states_and_reset_transitions, add_more_states_to_the_end

In [33]:
per_allele_per_run_per_split_prepared_models = dict()
per_allele_per_run_per_split_original_models = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_prepared_models[allele_name] = dict()
    per_allele_per_run_per_split_original_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
        per_allele_per_run_per_split_original_models[allele_name][run_num] = dict()
        for split_num, model in per_allele_per_run_per_split_models[allele_name][run_num].items():
            prepared_model = model.copy()
            prepared_model.name = prepared_model.name + "_iterations"
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model
            per_allele_per_run_per_split_original_models[allele_name][run_num][split_num] = prepared_model

In [34]:
initial_length_for_training = target_length_for_training

In [35]:
ADD_STATES_NUM = 4
for iteration in range(initial_length_for_training + 1,total_length+1):
    source_models = per_allele_per_run_per_split_prepared_models

    # copy models and rename
    per_allele_per_run_per_split_prepared_models = dict()
    for allele_name in ALLELES:
        print(f"allele {allele_name}")
        per_allele_per_run_per_split_prepared_models[allele_name] = dict()
        for run_num in range(NUM_RUNS):
            print(f"run {run_num}")
            per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
            for split_num, model in source_models[allele_name][run_num].items():
                print(f"split {split_num}")
                prepared_model = model.copy()
                old_states_names = [state.name for state in prepared_model.states]
                new_states_names =  state_names = ["s{:03d}".format(i) for i in range(len(prepared_model.states),len(prepared_model.states) + ADD_STATES_NUM)]
                original_model = per_allele_per_run_per_split_original_models[allele_name][run_num][split_num]
                new_model = add_more_states_to_the_end(prepared_model,
                                                                  ADD_STATES_NUM,
                                                                  new_sates_names =new_states_names,
                                                                  new_name = original_model.name + f"_to_length_{iteration}",
                                                                  new_probs="fair")

                print("Freezing the distribution for old states")
                print(f"old state {old_states_names}")
                frozen = []
                for state in new_model.states:
                    if (state.name in old_states_names) and state is not new_model.start and state is not new_model.end:
                        state.distribution.freeze()
                        frozen.append(state.name)
                print(f"freezed {frozen}")

                per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = new_model



    # make new data + 1 length
    per_allele_per_kfold_per_length_binders_train_filtered = dict()
    per_allele_per_kfold_per_length_binders_test_filtered = dict()

    for allele_name in ALLELES:
        per_allele_per_kfold_per_length_binders_train_filtered[allele_name] = dict()
        per_allele_per_kfold_per_length_binders_test_filtered[allele_name] = dict()

        for split_num in per_allele_per_kfold_per_length_binders_train[allele_name].keys():
            per_allele_per_kfold_per_length_binders_train_filtered[allele_name][split_num] = dict()
            per_allele_per_kfold_per_length_binders_test_filtered[allele_name][split_num] = dict()

            for target_length in per_allele_per_kfold_per_length_binders_train[allele_name][split_num].keys():
                data_array = per_allele_per_kfold_per_length_binders_train[allele_name][split_num][target_length]
                per_allele_per_kfold_per_length_binders_train_filtered[allele_name][split_num][target_length] = np.array([s[:iteration] for s in data_array ])

                data_array = per_allele_per_kfold_per_length_binders_test[allele_name][split_num][target_length]
                per_allele_per_kfold_per_length_binders_test_filtered[allele_name][split_num][target_length] = np.array([s[:iteration] for s in data_array ])
    # #train using new data
    # stop criteria correction
    target_length_for_training = iteration
    total_length = 9
    cutoff = 0.00005
    train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][9])
    ## Stop threshold Hardcoded here
    stop_threshold =  abs(cutoff * train_size * -22/ total_length  * target_length_for_training)
    per_allele_per_run_per_split_new_histories = dict()
    for allele_name in ALLELES:
        per_allele_per_run_per_split_new_histories[allele_name] = dict()
        for run_num in range(NUM_RUNS):
            per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train_filtered[allele_name]
            models, histories = train_model(allele_name,
                                            per_kfold_per_length_data,
                                            TARGET_LENGTHS,
                                            multiple_check_input=False,
                                            algorithm=algorithm,
                                            verbose=True,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test_filtered[allele_name],
                                            stop_threshold=stop_threshold,
                                            min_iterations= 5,
                                            prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num])
            per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
            per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
            # per_allele_per_run_per_split_models[allele_name][run_num] = models
            for split_num, model in models.items():
                per_name_models[model.name] = model
            # save models
            for split_num, model in models.items():
                history = histories[split_num]
                save_model(TARGET_PATH_TO_RESULTS, model,history)


allele HLA-A*02:01
run 0
split 0
Initial length 6
num states to add 4
After length 10
 N is  8
sum from start 2.0
Freezing the distribution for old states
old state ['s000', 's001', 's002', 's003', '0_run-9_model-start', '0_run-9_model-end']
freezed ['s000', 's001', 's002', 's003']
run 1
split 0
Initial length 6
num states to add 4
After length 10
 N is  8
sum from start 2.0
Freezing the distribution for old states
old state ['s000', 's001', 's002', 's003', '1_run-9_model-start', '1_run-9_model-end']
freezed ['s000', 's001', 's002', 's003']
allele HLA-A*03:01
run 0
split 0
Initial length 6
num states to add 4
After length 10
 N is  8
sum from start 2.0
Freezing the distribution for old states
old state ['s000', 's001', 's002', 's003', '0_run-9_model-start', '0_run-9_model-end']
freezed ['s000', 's001', 's002', 's003']
run 1
split 0
Initial length 6
num states to add 4
After length 10
 N is  8
sum from start 2.0
Freezing the distribution for old states
old state ['s000', 's001', 's002',

In [36]:
len(per_name_models.keys())

90

In [37]:
from hmm_visualization_methods import *

In [38]:
%%time

for i, model in enumerate(per_name_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
Wall time: 13.1 s


In [39]:
%%time

for i, model in enumerate(per_name_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
Wall time: 7min 39s


In [40]:
per_name_pyviz = dict()
for name, model in per_name_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0) (Delete zero edges)
Converting graph for model (1_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0) (Delete zero edges)
Converting graph for model (0_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_A_03_01-0) (Delete zero edges)
Converting graph for model (1_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_A_03_01-0) (Delete zero edges)
Converting graph for model (0_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_A_01_01-0) (Delete zero edges)
Converting graph for model (1_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_A_01_01-0) (Delete zero edges)
Converting graph for model (0_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_B_07_02-0) (Delete zero edges)
Converting graph for model (1_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_B_07_02-0) (Delete zero edges)
Converting graph for model (0_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_B_27_05-0) (Delete zero edges)
Converting graph for model (1_run-9-4-st-viter

## Aux viterbi

In [19]:
target_length_for_training = 9
total_length = 9
cutoff = 0.00005
train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][9])
## Stop threshold Hardcoded here
stop_threshold =  abs(cutoff * train_size * -22/ total_length  * target_length_for_training)
per_allele_per_run_per_split_new_histories = dict()

for allele_name in ALLELES:
    for run_num in range(NUM_RUNS):
        for split_num, model in per_allele_per_run_per_split_prepared_models[allele_name][run_num].items():
            prepared_model = model.copy()
            prepared_model.name = prepared_model.name + "_plus_viterbi"

            for state in prepared_model.states:
                if state is not prepared_model.start and state is not prepared_model.end:
                    state.distribution.thaw()
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

In [20]:
per_name_new_models = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train_filtered[allele_name]
        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        multiple_check_input=False,
                                        algorithm='viterbi',
                                        verbose=True,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test_filtered[allele_name],
                                        stop_threshold=stop_threshold,
                                        min_iterations= 5,
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num])
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_new_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)

split 0
train size 24309
Shuffling for fairness 
Simple training: Train Length 24309, init_batch_size None
[1] Improvement: -7838.327403575066	Time (s): 2.728
[2] Improvement: 835.8272084100172	Time (s): 2.342
[3] Improvement: 449.98972129053436	Time (s): 2.414
[4] Improvement: 292.90927951515187	Time (s): 2.349
[5] Improvement: 489.54910392372403	Time (s): 2.301
[6] Improvement: 436.81431718869135	Time (s): 2.215
[7] Improvement: 263.1048528418178	Time (s): 2.324
[8] Improvement: 242.6132020689547	Time (s): 2.315
[9] Improvement: 74.49806030571926	Time (s): 2.279
[10] Improvement: 155.47001084138174	Time (s): 2.462
[11] Improvement: 165.96640408260282	Time (s): 2.252
[12] Improvement: 22.17145641054958	Time (s): 2.275
Total Training Improvement: -4409.413786695921
Total Training Time (s): 37.8841
split 0
train size 24309
Shuffling for fairness 
Simple training: Train Length 24309, init_batch_size None
[1] Improvement: -7440.938518775045	Time (s): 2.396
[2] Improvement: 1722.8337270691

In [21]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
2
3
4
5
6
7
8
9
Wall time: 2.28 s


In [22]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
2
3
4
5
6
7
8
9
Wall time: 1min 53s


In [23]:
per_name_pyviz = dict()
for name, model in per_name_new_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-4-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_9_plus_viterbi) (Delete zero edges)
Converting graph for model (1_run-9-4-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_9_plus_viterbi) (Delete zero edges)
Converting graph for model (0_run-9-4-st-baum-welch-alg-5-min_iters_model-HLA_A_03_01-0_iterations_to_length_9_plus_viterbi) (Delete zero edges)
Converting graph for model (1_run-9-4-st-baum-welch-alg-5-min_iters_model-HLA_A_03_01-0_iterations_to_length_9_plus_viterbi) (Delete zero edges)
Converting graph for model (0_run-9-4-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01-0_iterations_to_length_9_plus_viterbi) (Delete zero edges)
Converting graph for model (1_run-9-4-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01-0_iterations_to_length_9_plus_viterbi) (Delete zero edges)
Converting graph for model (0_run-9-4-st-baum-welch-alg-5-min_iters_model-HLA_B_07_02-0_iterations_to_length_9_plus_viterbi) (Dele